# Orbit Event Parser — FLAN-T5-small Fine-tuning (v2)

Run every cell **1 → 10** in order. Do NOT skip cells.

| Cell | Purpose |
|------|---------|
| 1 | Install dependencies |
| 2 | Upload training data |
| 3 | Verify data |
| 4 | Load model & tokenizer |
| 5 | Tokenize dataset |
| 6 | Train |
| 7 | Test in-memory model (before saving) |
| 8 | **Save model** (pytorch format — cross-version safe) |
| 9 | Verify saved model from disk |
| 10 | Package & download |

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece protobuf safetensors

import transformers, datasets, torch
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"torch        : {torch.__version__}")
print(f"GPU          : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name     : {torch.cuda.get_device_name(0)}")
print("\n✅ Ready")

## Cell 2 — Upload Training Data

Upload **`event_training_data_final.jsonl`** from `nlp-server/data/`

In [ ]:
from google.colab import files
import os

DATA_FILE = 'event_training_data_final.jsonl'

if not os.path.exists(DATA_FILE):
    print("Upload event_training_data_final.jsonl ...")
    uploaded = files.upload()
    for fn in uploaded:
        if fn != DATA_FILE:
            os.rename(fn, DATA_FILE)
    print(f"✅ Saved as {DATA_FILE}")
else:
    print(f"✅ {DATA_FILE} already present")

print(f"Size: {os.path.getsize(DATA_FILE):,} bytes")

## Cell 3 — Verify Data

In [ ]:
import json

examples = []
with open(DATA_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(json.loads(line))

print(f"Total examples: {len(examples)}")
assert len(examples) >= 3000, f"Expected >=3000, got {len(examples)}"

bad = [i for i, e in enumerate(examples) if 'input' not in e or 'output' not in e]
assert not bad, f"Missing keys at: {bad[:5]}"

bad_fmt = [i for i, e in enumerate(examples) if '|' not in e['output']]
assert not bad_fmt, f"Non-pipe output at: {bad_fmt[:5]}"

print(f"\nSamples:")
for e in examples[:2]:
    print(f"  IN : {e['input'][:80]}")
    print(f"  OUT: {e['output'][:80]}")
    print()

rec = sum(1 for e in examples if 'recurrence: none' not in e['output'])
print(f"With recurrence: {rec}")
print("\n✅ Data OK")

## Cell 4 — Load Tokenizer & Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME     = "google/flan-t5-small"
OUTPUT_DIR     = "./event-parser-output"
MAX_INPUT_LEN  = 128
MAX_OUTPUT_LEN = 180
BATCH_SIZE     = 16
EPOCHS         = 20
LEARNING_RATE  = 3e-4
WARMUP_RATIO   = 0.08
WEIGHT_DECAY   = 0.01
PATIENCE       = 3

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"✅ Loaded ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params)")

## Cell 5 — Tokenize Dataset

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

raw = {'input': [e['input'] for e in examples],
       'output': [e['output'] for e in examples]}
ds = Dataset.from_dict(raw)

split1 = ds.train_test_split(test_size=0.20, seed=42)
split2 = split1['test'].train_test_split(test_size=0.50, seed=42)
train_ds, val_ds, test_ds = split1['train'], split2['train'], split2['test']
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")

def tokenize(batch):
    inputs = tokenizer(batch['input'], max_length=MAX_INPUT_LEN,
                       truncation=True, padding=False)
    labels = tokenizer(text_target=batch['output'], max_length=MAX_OUTPUT_LEN,
                       truncation=True, padding=False)
    inputs['labels'] = labels['input_ids']
    return inputs

tok_train = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(tokenize,   batched=True, remove_columns=val_ds.column_names)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model,
                                   label_pad_token_id=-100, padding=True)
print("✅ Tokenized")

## Cell 6 — Train

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"bf16: {use_bf16}  fp16: False")

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=30,
    fp16=False,
    bf16=use_bf16,
    predict_with_generate=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tok_train, eval_dataset=tok_val,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print(f"\nTraining ({EPOCHS} epochs max, patience={PATIENCE}) ...")
trainer.train()
print("\n✅ Training complete!")

## Cell 7 — Test In-Memory Model

Tests the trained model **before saving** to confirm the weights are good.  
If this cell fails, there is a training problem. If it passes but Cell 9 fails, there is a save/load problem.

In [ ]:
import torch

model.eval()

def generate_test(text, mdl=model, tok=tokenizer):
    inp = f"parse event: {text}"
    tokens = tok(inp, return_tensors='pt', max_length=128, truncation=True)
    if torch.cuda.is_available():
        tokens = {k: v.to(mdl.device) for k, v in tokens.items()}
    with torch.no_grad():
        out = mdl.generate(**tokens, max_new_tokens=150, num_beams=4,
                           early_stopping=True, no_repeat_ngram_size=3,
                           repetition_penalty=2.5)
    return tok.decode(out[0], skip_special_tokens=True)

tests = [
    "Meeting with John tomorrow at 3pm for 1 hour",
    "Yoga every Wednesday at 6pm",
    "Workshop 10am-11am today",
    "Lunch at noon with Sarah",
    "Daily standup at 10am",
]

all_ok = True
for t in tests:
    r = generate_test(t)
    ok = '|' in r and 'action:' in r.lower()
    print(f"{'✅' if ok else '❌'}  {t}")
    print(f"   → {r[:100]}")
    if not ok:
        all_ok = False

if all_ok:
    print("\n✅ In-memory model works — safe to save")
else:
    print("\n❌ Model produces bad output — do NOT proceed to save")
    print("   Check training data and re-run Cell 6")

## Cell 8 — Save Model (CRITICAL)

Saves as **pytorch_model.bin** (not safetensors) for cross-version compatibility.  
transformers 5.x safetensors has a weight-tying deduplication bug that breaks loading on 4.x.  

**DO NOT modify the model object before this cell. Run immediately after Cell 7.**

In [ ]:
import os, shutil, json, torch

SAVE_DIR = "./event-parser-final"

if os.path.exists(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)
os.makedirs(SAVE_DIR)

# ── Step 1: Save state_dict manually as pytorch_model.bin ──
# We clone every tensor to CPU float32 with independent memory,
# so tied weights become 4 separate tensors in the file.
# This avoids ALL weight-tying serialization issues.
state = {}
for k, v in model.state_dict().items():
    state[k] = v.detach().cpu().float().clone()

torch.save(state, os.path.join(SAVE_DIR, 'pytorch_model.bin'))
pt_size = os.path.getsize(os.path.join(SAVE_DIR, 'pytorch_model.bin'))
print(f"✅ pytorch_model.bin saved ({pt_size/1024/1024:.1f} MB, {len(state)} tensors)")

# ── Step 2: Save config.json (patched for inference + 4.x compat) ──
cfg = model.config.to_dict()
cfg['use_cache'] = True                  # enable KV-cache for inference
cfg['tie_word_embeddings'] = False        # all 4 tied tensors are saved independently
cfg.pop('transformers_version', None)     # remove version lock
# Remove any 5.x-only keys that might cause warnings on 4.x
for extra_key in ['_attn_implementation', '_attn_implementation_autoset']:
    cfg.pop(extra_key, None)

with open(os.path.join(SAVE_DIR, 'config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)
print("✅ config.json saved")

# ── Step 3: Save generation_config.json ──
gen_cfg = {
    "decoder_start_token_id": 0,
    "eos_token_id": 1,
    "pad_token_id": 0
}
with open(os.path.join(SAVE_DIR, 'generation_config.json'), 'w') as f:
    json.dump(gen_cfg, f, indent=2)
print("✅ generation_config.json saved")

# ── Step 4: Save tokenizer ──
tokenizer.save_pretrained(SAVE_DIR)
print("✅ Tokenizer saved")

# ── Step 5: Ensure spiece.model ──
spiece_dst = os.path.join(SAVE_DIR, 'spiece.model')
if not os.path.exists(spiece_dst):
    # Find it in cache or checkpoint dirs
    found = None
    for search_dir in [OUTPUT_DIR, '/root/.cache']:
        for root, dirs, flist in os.walk(search_dir):
            for fn in flist:
                if fn == 'spiece.model':
                    found = os.path.join(root, fn)
                    break
            if found:
                break
    if found:
        shutil.copy(found, spiece_dst)
        print(f"✅ spiece.model copied from {found}")
    else:
        from huggingface_hub import hf_hub_download
        src = hf_hub_download('google/flan-t5-small', 'spiece.model')
        shutil.copy(src, spiece_dst)
        print("✅ spiece.model downloaded")
else:
    print("✅ spiece.model present")

# ── Summary ──
print(f"\nFiles in {SAVE_DIR}:")
for fn in sorted(os.listdir(SAVE_DIR)):
    sz = os.path.getsize(os.path.join(SAVE_DIR, fn)) / 1024 / 1024
    print(f"  {fn:40s}  {sz:.1f} MB")
print("\n✅ Save complete — proceed to Cell 9 to verify")

## Cell 9 — Verify Saved Model From Disk

Loads the model **fresh from disk** and tests it.  
If this passes, the downloaded files are guaranteed to work on your Mac.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, T5ForConditionalGeneration
import torch, json, os

SAVE_DIR = "./event-parser-final"

# ── Method A: Load via AutoModel (normal path) ──
print("Loading from disk via AutoModelForSeq2SeqLM ...")
try:
    disk_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
    disk_mdl = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR)
    disk_mdl.eval()
    print("✅ AutoModel loaded")
except Exception as e:
    print(f"❌ AutoModel failed: {e}")
    print("Trying T5ForConditionalGeneration ...")
    with open(os.path.join(SAVE_DIR, 'config.json')) as f:
        cfg = json.load(f)
    from transformers import T5Config
    config = T5Config.from_dict(cfg)
    disk_mdl = T5ForConditionalGeneration(config)
    sd = torch.load(os.path.join(SAVE_DIR, 'pytorch_model.bin'), map_location='cpu')
    disk_mdl.load_state_dict(sd, strict=False)
    disk_mdl.eval()
    disk_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
    print("✅ Manual load succeeded")

# ── Compare weights ──
orig_v = model.state_dict()['shared.weight'][0,:5].detach().cpu().float().tolist()
disk_v = disk_mdl.state_dict()['shared.weight'][0,:5].detach().cpu().float().tolist()
match = all(abs(a-b) < 1e-4 for a, b in zip(orig_v, disk_v))
print(f"\nWeight check — shared.weight match: {match}")
if not match:
    print(f"  original: {[round(v,4) for v in orig_v]}")
    print(f"  disk:     {[round(v,4) for v in disk_v]}")

# ── Generate tests ──
tests = [
    ("Meeting with John tomorrow at 3pm for 1 hour", "action", None),
    ("Lunch at noon with Sarah",                      "action", None),
    ("Conference at HKUST next week",                 "location", "HKUST"),
    ("Yoga class at the gym Friday 6pm",              "location", "gym"),
    ("Workshop today 2pm for 3 hours",                "duration", "3 hour"),
    ("Yoga every Wednesday at 6pm",                   "recurrence", None),
    ("Daily standup at 10am",                         "recurrence", None),
    ("Weekly team meeting every Monday at 9am",       "recurrence", None),
    ("Tennis every Friday at 5pm",                    "recurrence", None),
    ("Coffee with mentor tomorrow 3pm at Starbucks",  "location", "Starbucks"),
    ("Meeting 10am-11:20am tomorrow",                 "duration", None),
    ("Workshop 3pm-5pm today",                        "duration", None),
    ("Birthday party next Saturday at 5pm",           "action", None),
    ("Team standup next Monday at 10am",              "action", None),
    ("Doctor at Central Hospital tomorrow 2pm",       "location", "Hospital"),
    ("Gym each Tuesday at 7am",                       "recurrence", None),
    ("Monthly review first Monday at 2pm",            "recurrence", None),
    ("Dance class each Saturday at 10am",             "recurrence", None),
    ("Project review Friday 2pm in Room 301 for 2h", "location", "Room 301"),
    ("Call 9am-9:30am on Friday",                     "duration", None),
]

passed = failed = 0
print(f"\n{'#':>3}  {'Input':45}  {'St':8}  Output")
print("-" * 115)

for i, (text, field, expect) in enumerate(tests, 1):
    tokens = disk_tok(f"parse event: {text}", return_tensors='pt',
                      max_length=128, truncation=True)
    with torch.no_grad():
        out = disk_mdl.generate(**tokens, max_new_tokens=150, num_beams=4,
                                early_stopping=True, no_repeat_ngram_size=3,
                                repetition_penalty=2.5)
    r = disk_tok.decode(out[0], skip_special_tokens=True)

    ok = '|' in r and 'action:' in r.lower()
    if expect:
        ok = ok and expect.lower() in r.lower()
    if not r or len(r) < 5:
        ok = False

    st = '✅' if ok else '❌'
    if ok: passed += 1
    else:  failed += 1
    print(f"{i:>3}  {text[:44]:45}  {st}       {r[:60]}")

total = len(tests)
rate = passed / total * 100
print(f"\nResult: {passed}/{total} passed ({rate:.0f}%)")
if rate >= 85:
    print("✅ EXCELLENT — safe to download")
elif rate >= 65:
    print("🟡 ACCEPTABLE — download with caution")
else:
    print("🔴 POOR — do NOT download, check training")

## Cell 10 — Package & Download

After downloading, on your Mac:
```bash
cd /path/to/nlp-server/models
rm -rf event-parser
unzip ~/Downloads/event-parser-v3.zip -d event-parser
python3 server.py
```

In [ ]:
import os, zipfile
from google.colab import files

SAVE_DIR = "./event-parser-final"
ZIP_PATH = '/content/event-parser-v3.zip'

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(SAVE_DIR)):
        fp = os.path.join(SAVE_DIR, fn)
        if os.path.isfile(fp):
            zf.write(fp, fn)
            print(f"  + {fn:40s} ({os.path.getsize(fp)/1024/1024:.1f} MB)")

print(f"\n✅ {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1024/1024:.1f} MB)")
files.download(ZIP_PATH)